In [15]:
#LETS ADD TWO MORE datasets for comparision - era5L ordinary kriged and station ordinary kriged


## LIBRARIES
import pandas as pd
import numpy as np
import glob
import os
import rasterio
import HydroErr as he
import concurrent.futures
from pyproj import Transformer

## PATHS
insitu_csv    = r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv"
train_stations = r"..\Data\Processed\Split_stations\train_stations.csv"
test_stations  = r"..\Data\Processed\Split_stations\test_stations.csv"
chirps_path    = r"..\Data\Datasets\CHIRPS_prec"
persiann_path  = r"..\Data\Datasets\PERSIANN_prec"
mswep_path     = r"..\Data\Datasets\MSWEP_prec"
har_path       = r"..\Data\Datasets\HAR_prec"
era5l_path     = r"..\Data\Datasets\ERA5L_prec"
ds_path        = r"..\Data\Processed\PredictedRasters\Final_rasters"
ds_cal_path    = r"..\Data\Processed\Calibration_kriging\Calibrated_rasters"
era5K_path = r"..\Data\Processed\Kriging_Predictors\Kriged_rasters"
output_dir = r"..\Data\Processed\FinalValidation"
os.makedirs(output_dir, exist_ok= True)


## PARAMETERS
daily_datasets   = ['CHIRPS', 'PERSIANN', 'HAR', 'MSWEP', 'ERA5L', 'ERA5K']
monthly_datasets = ['DS', 'DS_Cal']
datasets         = daily_datasets + monthly_datasets

start_year, start_month = 2001, 1
end_year,   end_month   = 2005, 12
start_date = pd.Timestamp(f"{start_year}-{start_month:02d}-01")
end_date   = pd.Timestamp(f"{end_year}-{end_month:02d}-01") + pd.offsets.MonthEnd(0)

dataset_paths = {
    'CHIRPS':   chirps_path,
    'PERSIANN': persiann_path,
    'MSWEP':    mswep_path,
    'HAR':      har_path,
    'ERA5L':    era5l_path,
    'ERA5K': era5K_path,
    'DS':       ds_path,
    'DS_Cal':   ds_cal_path,
}

In [16]:
# %%
insitu_raw = pd.read_csv(insitu_csv)
insitu_raw["date"] = pd.to_datetime(insitu_raw[["year", "month", "day"]])
insitu_raw = insitu_raw.rename(columns={
    "lat": "y", "long": "x", "rainfall_sum": "insitu_prec"
})
insitu_raw = insitu_raw[insitu_raw["insitu_prec"] >= 0]

insitu_monthly = (
    insitu_raw
    .groupby(["index_no", "y", "x", "year", "month"])
    .agg(
        insitu_monthly_total=("insitu_prec", "sum"),
        valid_days          =("insitu_prec", "count")
    )
    .reset_index()
)

insitu_monthly = insitu_monthly[
    (insitu_monthly["year"] >= start_year) &
    (insitu_monthly["year"] <= end_year)
].copy()
insitu_monthly[["x", "y"]] = insitu_monthly[["x", "y"]].round(5)

# Train / test split
train_ids = set(pd.read_csv(train_stations)["index_no"])
test_ids  = set(pd.read_csv(test_stations)["index_no"])
insitu_train_monthly = insitu_monthly[insitu_monthly["index_no"].isin(train_ids)].copy()
insitu_test_monthly  = insitu_monthly[insitu_monthly["index_no"].isin(test_ids)].copy()

In [17]:
# %%
def extract_date(filepath, dataset):
    """Parse date from filename for both daily and monthly naming conventions."""
    fname = os.path.basename(filepath)
    if dataset in ['DS', 'DS_Cal']:
        # e.g. Calibrated_Raster_2001_01.tif → fname[-11:-4] = '2001_01'
        return pd.to_datetime(fname[-11:-4], format='%Y_%m')
    else:
        # e.g. CHIRPS_20010115.tif → fname[-12:-4] = '20010115'
        return pd.to_datetime(fname[-12:-4], format='%Y%m%d')


def build_dataset_results(dataset_paths, datasets):
    """Build {dataset}_files dict pre-filtered to analysis period."""
    dataset_results = {}
    for dataset in datasets:
        files = sorted(glob.glob(os.path.join(dataset_paths[dataset], "*.tif")))
        filtered = []
        for f in files:
            try:
                d = extract_date(f, dataset)
                if start_date <= d <= end_date:
                    filtered.append(f)
            except Exception as e:
                print(f"  Skipping {os.path.basename(f)}: {e}")
        dataset_results[f'{dataset}_files'] = filtered
        print(f"{dataset}: {len(filtered)} files in period")
    return dataset_results


# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn = (
    insitu_test_monthly[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points = list(zip(unique_stn['x'], unique_stn['y']))  # (lon, lat)

dataset_results = build_dataset_results(dataset_paths, datasets)

CHIRPS: 1826 files in period
PERSIANN: 1826 files in period
HAR: 1826 files in period
MSWEP: 1826 files in period
ERA5L: 1825 files in period
ERA5K: 1825 files in period
DS: 60 files in period
DS_Cal: 60 files in period


In [18]:
# %%
def _sample_file(file, dataset, stn_points, unique_stn):
    """Sample all station points from a single raster file."""
    date_obj = extract_date(file, dataset)
    try:
        with rasterio.open(file) as src:
            if src.crs.to_epsg() != 4326:
                transformer = Transformer.from_crs(
                    "EPSG:4326", src.crs, always_xy=True
                )
                pts = [transformer.transform(x, y) for x, y in stn_points]
            else:
                pts = stn_points

            sampled = list(src.sample(pts))

        return [
            {
                'date':              date_obj,
                'stn_id':            unique_stn.iloc[i]['index_no'],
                f'{dataset}_prec':   max(v[0], 0),
            }
            for i, v in enumerate(sampled)
        ]
    except Exception as e:
        print(f"  Error processing {os.path.basename(file)}: {e}")
        return []


def extract_and_aggregate_daily(dataset_results, stn_points, unique_stn, datasets):
    """
    For daily-file datasets:
      1. Sample each file concurrently
      2. Aggregate to monthly SUMS per dataset independently
      3. Merge datasets at monthly level (outer join to preserve coverage gaps)
    """
    df_final = None

    for dataset in datasets:
        files = dataset_results.get(f'{dataset}_files', [])
        if not files:
            print(f"Warning: no files found for {dataset}, skipping.")
            continue
        print(f"Processing {dataset} ({len(files)} daily files)...")

        records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = executor.map(
                lambda f: _sample_file(f, dataset, stn_points, unique_stn), files
            )
            for result in futures:
                records.extend(result)

        df_dataset = pd.DataFrame(records)
        if df_dataset.empty:
            print(f"  Warning: no data extracted for {dataset}")
            continue

        # Aggregate daily → monthly sum BEFORE merging
        df_dataset['date'] = pd.to_datetime(df_dataset['date'])
        df_monthly = (
            df_dataset
            .assign(
                year  = lambda d: d['date'].dt.year,
                month = lambda d: d['date'].dt.month
            )
            .groupby(['stn_id', 'year', 'month'])[f'{dataset}_prec']
            .sum()
            .reset_index()
        )

        df_final = (
            df_monthly if df_final is None
            else pd.merge(df_final, df_monthly,
                          on=['stn_id', 'year', 'month'], how='outer')
        )
        print(f"  → merged shape: {df_final.shape}")

    return df_final


def extract_monthly_datasets(dataset_results, stn_points, unique_stn, datasets):
    """
    For already-monthly datasets (DS, DS_Cal):
      Sample each file concurrently and merge at monthly level.
    """
    df_final = None

    for dataset in datasets:
        files = dataset_results.get(f'{dataset}_files', [])
        if not files:
            print(f"Warning: no files found for {dataset}, skipping.")
            continue
        print(f"Processing {dataset} ({len(files)} monthly files)...")

        records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            for result in executor.map(
                lambda f: _sample_file(f, dataset, stn_points, unique_stn), files
            ):
                records.extend(result)

        df_dataset = pd.DataFrame(records)
        if df_dataset.empty:
            print(f"  Warning: no data extracted for {dataset}")
            continue

        df_dataset['date'] = pd.to_datetime(df_dataset['date'])
        df_dataset = df_dataset.assign(
            year  = lambda d: d['date'].dt.year,
            month = lambda d: d['date'].dt.month
        ).drop(columns='date')

        df_final = (
            df_dataset if df_final is None
            else pd.merge(df_final, df_dataset,
                          on=['stn_id', 'year', 'month'], how='outer')
        )
        print(f"  → merged shape: {df_final.shape}")

    return df_final

In [19]:
# %%
df_grid_daily   = extract_and_aggregate_daily(
    dataset_results, stn_points, unique_stn, daily_datasets
)

df_grid_monthly_ds = extract_monthly_datasets(
    dataset_results, stn_points, unique_stn, monthly_datasets
)

Processing CHIRPS (1826 daily files)...
  → merged shape: (1080, 4)
Processing PERSIANN (1826 daily files)...
  → merged shape: (1080, 5)
Processing HAR (1826 daily files)...
  → merged shape: (1080, 6)
Processing MSWEP (1826 daily files)...
  → merged shape: (1080, 7)
Processing ERA5L (1825 daily files)...
  → merged shape: (1080, 8)
Processing ERA5K (1825 daily files)...
  → merged shape: (1080, 9)
Processing DS (60 monthly files)...
  → merged shape: (1080, 4)
Processing DS_Cal (60 monthly files)...
  → merged shape: (1080, 5)


In [20]:
# %%
# Merge daily-sourced and monthly-sourced grid data
df_grid_all = pd.merge(
    df_grid_daily,
    df_grid_monthly_ds,
    on=['stn_id', 'year', 'month'],
    how='inner'   # only keep months present in all datasets
)

# Merge with insitu test observations
df_validation = pd.merge(
    insitu_test_monthly.rename(columns={'index_no': 'stn_id'}),
    df_grid_all,
    on=['stn_id', 'year', 'month'],
    how='inner'
)

# Optional: filter to months with sufficient insitu coverage
df_validation = df_validation[df_validation['valid_days'] >= 25].copy()

print(f"Validation df shape: {df_validation.shape}")
print(f"Stations: {df_validation['stn_id'].nunique()}")
print(f"Columns: {df_validation.columns.tolist()}")
print(df_validation.head())

df_validation.to_csv(os.path.join(output_dir,r"validation_df.csv"), index= False)

Validation df shape: (1035, 15)
Stations: 18
Columns: ['stn_id', 'y', 'x', 'year', 'month', 'insitu_monthly_total', 'valid_days', 'CHIRPS_prec', 'PERSIANN_prec', 'HAR_prec', 'MSWEP_prec', 'ERA5L_prec', 'ERA5K_prec', 'DS_prec', 'DS_Cal_prec']
   stn_id         y         x  year  month  insitu_monthly_total  valid_days  \
0     102  29.55744  80.41317  2001      1                  36.0          31   
1     102  29.55744  80.41317  2001      2                  43.0          28   
2     102  29.55744  80.41317  2001      3                  43.0          31   
3     102  29.55744  80.41317  2001      4                  60.7          30   
4     102  29.55744  80.41317  2001      5                 232.6          31   

   CHIRPS_prec  PERSIANN_prec    HAR_prec  MSWEP_prec  ERA5L_prec  ERA5K_prec  \
0    10.045458      26.930243   29.309715      5.6250   38.278797   40.044651   
1    27.383974      14.299629   71.920990      1.0625   25.778899   26.395853   
2    42.603962      27.285402  138

In [21]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_prec):
    df_accuracy = pd.DataFrame()

    datasets = ['CHIRPS', 'PERSIANN', 'HAR', 'MSWEP', 'ERA5L', 'ERA5K', 'DS', 'DS_Cal']

    for dataset in datasets:
        dataset_accuracy = {'dataset': dataset}

        #Convert to array
        prec_grid_array = dataframe_prec[f'{dataset}_prec'].values
        prec_insitu_array = dataframe_prec['insitu_monthly_total'].values
    
        #drop null values, and negative values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(prec_grid_array),
            ~np.isnan(prec_insitu_array),
            prec_grid_array >=  0,
            prec_insitu_array >= 0
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        prec_grid_array = prec_grid_array[valid_indices]
        prec_insitu_array = prec_insitu_array[valid_indices]

        #correlation coefficient
        correlation = he.spearman_r(prec_grid_array, prec_insitu_array)
        dataset_accuracy['correlation'] = correlation

        #NRMSE
        nrmse = he.nrmse_mean(prec_grid_array, prec_insitu_array)
        dataset_accuracy['nrmse'] = nrmse

        #Percent bias
        pbias = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
        dataset_accuracy['pbias'] = pbias

        #Relative standard deviation 
        rsd = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
        dataset_accuracy['rsd'] = rsd

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    #Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    #converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    #get cpi for each (linear)
    df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
    return df_accuracy_final

#get daily data accuracy
df_accuracy = calculate_accuracy(df_validation)
df_accuracy

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,CHIRPS,0.748627,3.0,0.890652,0.724501,-19.417312,0.732370,1035
1,PERSIANN,1.276112,6.0,0.888993,0.905013,-30.025842,0.577138,1035
2,HAR,1.526781,7.0,0.821765,0.940364,-9.837333,0.792057,1035
3,MSWEP,4.000000,8.0,0.821174,1.546156,-90.076246,0.093958,1035
4,ERA5L,1.157284,5.0,0.846930,0.879359,21.348403,1.054525,1035
5,ERA5K,1.057151,4.0,0.852353,0.861714,20.315939,1.045323,1035
6,DS,0.213731,1.0,0.915003,0.638879,-8.879201,0.778467,1035
7,DS_Cal,0.228001,2.0,0.915856,0.643518,-10.738936,0.782547,1035


In [22]:
# determine the same but for training stations data

# Unique TRAIN stations (mirrors the test station setup)
unique_stn_train = (
    insitu_train_monthly[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_train = list(zip(unique_stn_train['x'], unique_stn_train['y']))  # (lon, lat)

# Extract grid values at training station locations
df_grid_daily_train = extract_and_aggregate_daily(
    dataset_results, stn_points_train, unique_stn_train, daily_datasets
)

df_grid_monthly_ds_train = extract_monthly_datasets(
    dataset_results, stn_points_train, unique_stn_train, monthly_datasets
)

# Merge daily-sourced and monthly-sourced grid data
df_grid_all_train = pd.merge(
    df_grid_daily_train,
    df_grid_monthly_ds_train,
    on=['stn_id', 'year', 'month'],
    how='inner'
)

# Merge with insitu train observations
df_validation_train = pd.merge(
    insitu_train_monthly.rename(columns={'index_no': 'stn_id'}),
    df_grid_all_train,
    on=['stn_id', 'year', 'month'],
    how='inner'
)

# Filter to months with sufficient insitu coverage (consistent with test set)
df_validation_train = df_validation_train[df_validation_train['valid_days'] >= 25].copy()

print(f"Train validation df shape: {df_validation_train.shape}")
print(f"Stations: {df_validation_train['stn_id'].nunique()}")
print(f"Columns: {df_validation_train.columns.tolist()}")
print(df_validation_train.head())

df_validation_train.to_csv(os.path.join(output_dir, "validation_train_df.csv"), index=False)

# Calculate accuracy for training stations
df_accuracy_train = calculate_accuracy(df_validation_train)
df_accuracy_train

Processing CHIRPS (1826 daily files)...
  → merged shape: (3000, 4)
Processing PERSIANN (1826 daily files)...
  → merged shape: (3000, 5)
Processing HAR (1826 daily files)...
  → merged shape: (3000, 6)
Processing MSWEP (1826 daily files)...
  → merged shape: (3000, 7)
Processing ERA5L (1825 daily files)...
  → merged shape: (3000, 8)
Processing ERA5K (1825 daily files)...
  → merged shape: (3000, 9)
Processing DS (60 monthly files)...
  → merged shape: (3000, 4)
Processing DS_Cal (60 monthly files)...
  → merged shape: (3000, 5)
Train validation df shape: (2883, 15)
Stations: 50
Columns: ['stn_id', 'y', 'x', 'year', 'month', 'insitu_monthly_total', 'valid_days', 'CHIRPS_prec', 'PERSIANN_prec', 'HAR_prec', 'MSWEP_prec', 'ERA5L_prec', 'ERA5K_prec', 'DS_prec', 'DS_Cal_prec']
   stn_id         y         x  year  month  insitu_monthly_total  valid_days  \
0     107  29.84496  80.54539  2001      1                  41.4          31   
1     107  29.84496  80.54539  2001      2              

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,CHIRPS,1.259252,3.0,0.885605,0.695491,-8.470262,0.817897,2883
1,PERSIANN,1.944658,6.0,0.875679,0.976946,-24.312407,0.592280,2883
2,HAR,2.064657,7.0,0.811747,1.238164,20.191208,1.090640,2883
3,MSWEP,3.973227,8.0,0.816623,1.563849,-89.328496,0.096058,2883
4,ERA5L,1.849196,5.0,0.861077,1.028958,33.662997,1.132474,2883
5,ERA5K,1.839019,4.0,0.862898,1.027305,33.948100,1.130523,2883
6,DS,0.156124,2.0,0.983658,0.207276,0.268607,0.936818,2883
7,DS_Cal,0.013904,1.0,0.993863,0.110473,-1.506878,0.965943,2883
